# Notebook 6 — Dashboard de celeridade (Streamlit)

**Objectivo:** interface para gestão — pendência prevista, prazo típico, faixa de celeridade — **sem re-treino**. Linguagem de gestão em todo o painel; nunca a notação $\hat S(t\mid x)$ do texto académico.

**App:** `dashboard/app.py`. Este caderno gera os catálogos (`tabelas/dash_*.csv`) e documenta a arquitectura. O Streamlit **não** corre dentro do Jupyter.

**Horizontes de exibição** (distintos da avaliação académica do Notebook 3): 6 meses, 1 ano, 2 anos, 5 anos e **mais de 7 anos** ($t=2555$ dias).

**Secção da dissertação:** Demonstração aplicada (Capítulo 5). **Não** faz parte da validação estatística formal.


## 0. Configuração


In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src").is_dir() else HERE.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dash_aft import (
    caminho_booster, caminho_encoder, caminho_meta,
    carregar_modelo_aft, construir_artefactos_dash, dir_tab,
)
from src.data_utils import project_root

ROOT = project_root()
print("Booster:", caminho_booster(), caminho_booster().is_file())
print("Encoder:", caminho_encoder(), caminho_encoder().is_file())
print("Meta   :", caminho_meta(), caminho_meta().is_file())
pacote = carregar_modelo_aft()
print("Distribuição", pacote["dist"], "σ", round(pacote["sigma"], 3))


Booster: G:\O meu disco\Main\Cursos\ISCTE\MScBA\Dissertação\XGBOOST\XGBOOST\modelo\xgboost_aft_modelo_final.json True
Encoder: G:\O meu disco\Main\Cursos\ISCTE\MScBA\Dissertação\XGBOOST\XGBOOST\modelo\xgboost_aft_encoder.joblib True
Meta   : G:\O meu disco\Main\Cursos\ISCTE\MScBA\Dissertação\XGBOOST\XGBOOST\modelo\xgboost_aft_meta.json True
Distribuição normal σ 0.963


## 1. Catálogos do painel

Gera `dash_catalogo_*.csv`, prazos por célula classe×órgão×ano, percentis de faixa, exemplos e anomalias. Uma previsão AFT por célula — barato, porque μ é constante na célula.


In [2]:
paths = construir_artefactos_dash(forcar=True)
for k, p in paths.items():
    print(f"{k:12s}  {p.name}  {p.stat().st_size/1e6:.2f} MB")


A ler o universo (só colunas de capa)…
Células classe × órgão × ano: 15 576
A amostrar processos da base de teste (seed=42)…
Artefactos do painel gravados em tabelas/dash_*.csv
volume        dash_volume_classe_orgao_ano.csv  1.10 MB
prazos        dash_prazos_celulas.csv  3.29 MB
percentis     dash_percentis_classe_ano.csv  0.02 MB
catalogo      dash_catalogo_classe_orgao.csv  0.14 MB
exemplos      dash_exemplos_processos.csv  0.00 MB
anomalias     dash_anomalias.csv  0.10 MB


## 2. Arquitectura da interface

Páginas em `dashboard/app.py`:

1. **Consulta por combinação** — dropdowns de órgão, classe e ano; curva com linhas verticais nos horizontes de exibição e rótulo percentual; cartões de pendência (6 meses, 1 ano, 2 anos, 5 anos, mais de 7 anos); semáforo de faixa.
2. **Comparar órgãos** — uma classe fixa, vários órgãos; overlay das curvas. Não se misturam classes.
3. **Tendência no tempo** — uma classe e um órgão fixos, 2015–2025, com a ressalva da Figura 18.
4. **Evolução no tempo (visão global)** — distingue coortes com follow-up suficiente (até 2023) das de 2024–2025.
5. **Consulta por número CNJ** — API DataJud em tempo real; ano fora de 2015–2025 usa o ano treinado mais próximo (2025) como *proxy*, com aviso visível; classes fora do top 95% mapeiam para «Outras classes».

Arranque, na raiz do repositório:

```text
streamlit run dashboard/app.py
```


In [3]:
display(Markdown('''
**Nota para o Capítulo 5.** O dashboard é uma demonstração de uso gerencial do *booster* já validado. Não gera métricas novas de C-index ou IBS. A consulta de processos de 2026 é uma **extrapolação** (dummy de ano = 2025) e deve ser descrita como tal na Discussão.
'''))
print("Tabelas do painel em", dir_tab())


<IPython.core.display.Markdown object>
Tabelas do painel em G:\O meu disco\Main\Cursos\ISCTE\MScBA\Dissertação\XGBOOST\XGBOOST\tabelas
